# FADE — Immediate retry (NULL baseline)

On a wrong answer, blindly re-sample the SAME prompt at temperature 0.8 — no diagnosis, no queues, no typed exemplars. Isolates whether retrying at all helps vs whether *diagnosis* helps. Writes `store_immediate/`.

**Sidebar:** GPU on · Internet on · Secret `HF_TOKEN` (this account's, Llama-2 approved) (the shared eval order is committed in the repo — nothing to attach).

The shared eval order is restored from that dataset so THIS arm evaluates the identical 1,319 problems as every other arm, on any account.


In [1]:
# 1. Deps + clone
import os, shutil
!pip -q install -U "transformers>=4.40" accelerate sentence-transformers sympy scikit-learn datasets scipy 2>/dev/null | tail -1
REPO='/kaggle/working/fade'
if os.path.exists(REPO): shutil.rmtree(REPO)
!git clone --depth 1 https://github.com/S2V3/fade.git {REPO}
print('cloned')


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 30.6 MB/s eta 0:00:00
Cloning into '/kaggle/working/fade'...
remote: Enumerating objects: 20, done.
remote: Counting objects: 100% (20/20), done.
remote: Compressing objects: 100% (20/20), done.
remote: Total 20 (delta 0), reused 10 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (20/20), 352.42 KiB | 6.65 MiB/s, done.
cloned


In [2]:
# 2. The shared eval order is COMMITTED IN THE REPO (artifacts/), so it arrives with the clone.
import os
adir=f'{REPO}/artifacts'
files=os.listdir(adir) if os.path.exists(adir) else []
ok=any('evalorder' in f for f in files)
assert ok, ('artifacts/ not in the repo. Prime it once locally (--build-artifacts-only) and '
            'commit artifacts/ to the repo, then re-clone. Got: '+str(files))
print('shared eval order present:', files)

shared eval order present: ['seeds_evalorder_pool2000_test.json']


In [3]:
# 3. Confirm HF token
from kaggle_secrets import UserSecretsClient
print('HF_TOKEN length', len(UserSecretsClient().get_secret('HF_TOKEN')))


HF_TOKEN length 37


In [4]:
# 4. Config — MUST be identical across ALL arms/accounts
MODEL='meta-llama/Llama-2-7b-chat-hf'; N=1319; BUDGET=8; TOK=320
print(MODEL, N, BUDGET, TOK)


meta-llama/Llama-2-7b-chat-hf 1319 8 320


In [ ]:
# 5. RUN this arm (immediate). Resumable: re-run to continue after a disconnect.
!cd {REPO} && python kaggle_run.py --n-problems {N} --strategy 2 --split test \
    --model {MODEL} --retrieval 3stage --retry-mode immediate --run-name immediate \
    --exemplar-budget {BUDGET} --max-new-tokens {TOK} --show-every 50 --secret-name HF_TOKEN


Torch OK
 sentence-transformers OK
  generation module version: v5-hashline
  v5-hashline ACTIVE | N_SHOTS=8 rep_penalty=1.15 no_repeat_ngram=6
  ban_strings=['\\begin{code}', '\\end{code}', '```'] | stop_markers=['\nQuestion:', '\nQ:', '\nExample', '\nProblem:']
  preprocessing: normalize_traces=True
  exemplar budget: 8 (enforced across all arms)
  model: meta-llama/Llama-2-7b-chat-hf
  HF token loaded from Kaggle Secret 'HF_TOKEN'
  HF identity: S2V3 (type=user)
  gated-access probe OK: meta-llama/Llama-2-7b-chat-hf reachable (16 files)

Loading GSM8K (train for seeds, test for eval)...
README.md: 7.93kB [00:00, 3.39MB/s]
main/train-00000-of-00001.parquet: 100%|███| 2.31M/2.31M [00:00<00:00, 3.59MB/s]
main/test-00000-of-00001.parquet: 100%|██████| 419k/419k [00:00<00:00, 1.97MB/s]
Generating test split: 100%|█████| 1319/1319 [00:00<00:00, 214771.03 examples/s]
  source: openai/gsm8k [train]
  train: 2100 problems ready (preprocessed) | dropped 0 unparseable
  source: openai/gsm8k [t

In [ ]:
# 6. Results
import json; s=json.load(open(f'{REPO}/store_immediate/run_summary.json'))
print('IMMEDIATE  pass1=%.1f%%  final=%.1f%%  recovered=%d  pool=%d'%(
      100*s['pass1_accuracy'],100*s['final_accuracy'],s['recovered_by_retry'],s['final_pool_size']))
for it in s['iterations']:
    print('  iter%d MEDIUM %d/%d  BAD %d/%d  gap %+.3f'%(it['iter'],
          it['MEDIUM']['newly_correct'],it['MEDIUM']['retried'],
          it['BAD']['newly_correct'],it['BAD']['retried'],it['retry_worthiness_gap']))
print(open(f'{REPO}/store_immediate/report.md').read())


In [ ]:
# 7. Export
import shutil; shutil.make_archive('/kaggle/working/store_immediate','zip',f'{REPO}/store_immediate')
print('wrote /kaggle/working/store_immediate.zip — download for the combined comparison')


Independent run — identical whether alone or parallel, given the shared eval order. Collect `store_immediate.zip`.
